# SupplyForge AI 2.0: KAN-29 - Diaper Attribute Extraction



Tài liệu này thực hiện trích xuất cấu trúc các thuộc tính của tã/bỉm (Size, Type, Pack, MinKG, MaxKG) từ tên sản phẩm thô (`productName`) và tên biến thể phân loại SKU (`skuName`) của các giao dịch thô.



Quy trình bóc tách sử dụng Regular Expressions kết hợp chuẩn hóa và điền khuyết tự động theo bảng tra cứu nghiệp vụ nhằm sinh ra các đặc trưng sẵn sàng cho học máy (machine-learning-ready features) phục vụ mô hình dự báo.


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):

1. **Dữ liệu nguồn**: Trích xuất toàn bộ biến thể SKU tã/bỉm độc nhất (itemId x modelId) trong tập đơn hàng `order_1_cleaned.csv`.

2. **Size tiêu chuẩn**: Trích xuất size `NB`, `S`, `M`, `L`, `XL`, `XXL`, `XXXL`, `XXXXL`.

3. **Type tiêu chuẩn**: Phân loại thành tã dán (`TAPE`) hoặc tã quần (`PANTS`).

4. **Pack size**: Số lượng miếng tã trong bịch. Phải xử lý đúng các trường hợp cộng miếng (ví dụ: `38+6` -> `44` miếng).

5. **Cân nặng khuyến nghị (MinKG/MaxKG)**:

   - Trích xuất trực tiếp nếu xuất hiện trong tên (ví dụ: `12-17kg`).

   - Điền mặc định theo bảng tra cứu (Size Lookup Table) nếu bị khuyết.


In [1]:
import pandas as pd

import numpy as np

import json

import re

import os



DATA_DIR = '/home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH'

order_path = os.path.join(DATA_DIR, 'order_1_cleaned.csv')



print("Đang tải dữ liệu đơn hàng sạch...")

df_orders = pd.read_csv(order_path, low_memory=False)

print(f"Tổng số dòng đơn hàng sạch: {len(df_orders)}")


Đang tải dữ liệu đơn hàng sạch...


Tổng số dòng đơn hàng sạch: 39369


In [2]:
print("Đang trích xuất danh sách tất cả các biến thể tã/bỉm độc nhất (itemId * modelId)...")

items_list = []

for idx, row in df_orders.iterrows():

    data_str = row['data']

    if pd.notna(data_str):

        try:

            data_dict = json.loads(data_str)

            for item in data_dict.get('item_list', []):

                item_name = item.get('item_name', '')

                model_name = item.get('model_name', '')

                item_id = str(item.get('item_id', ''))

                model_id = str(item.get('model_id', ''))

                

                # Chỉ lọc các sản phẩm là tã/bỉm thực tế

                if any(k in item_name.lower() for k in ['tã', 'bỉm', 'tả', 'huggies', 'moony', 'merries']):

                    # Loại bỏ các sản phẩm phụ kiện khác

                    if not any(k in item_name.lower() for k in ['kem', 'kệ', 'túi', 'khăn', 'quần áo', 'mustela']):

                        items_list.append({

                            'itemId': item_id,

                            'modelId': model_id,

                            'productName': item_name,

                            'skuName': model_name

                        })

        except:

            pass



df_skus = pd.DataFrame(items_list).drop_duplicates(subset=['itemId', 'modelId'])

print(f"Tổng số biến thể SKU tã/bỉm tìm thấy: {len(df_skus)}")

print(f"Số lượng parent itemId độc nhất: {df_skus['itemId'].nunique()}")


Đang trích xuất danh sách tất cả các biến thể tã/bỉm độc nhất (itemId * modelId)...


Tổng số biến thể SKU tã/bỉm tìm thấy: 157
Số lượng parent itemId độc nhất: 19


In [3]:
# Bảng khoảng cân nặng mặc định theo size

size_weight_lookup = {

    'NB': (0.0, 5.0),

    'S': (4.0, 8.0),

    'M': (6.0, 11.0),

    'L': (9.0, 14.0),

    'XL': (12.0, 17.0),

    'XXL': (15.0, 25.0),

    'XXXL': (18.0, 35.0),

    'XXXXL': (22.0, 40.0)

}



def clean_text(text):

    t = str(text).lower()

    # Loại bỏ các từ gây nhiễu để các token size và pack được tách biệt rõ ràng

    words_to_remove = ['miếng', 'cái', 'quần', 'dán', 'tã', 'bỉm', 'tả', 'dùng', 'thử', 'chính', 'hãng', 'công', 'ty', 'cty', 'natural', 'super', 'dry', 'supdry', 'moony', 'merries', 'huggies', 'gooby', 'yubest', 'pamper', 'bobby', 'khác', 'bé', 'sơ sinh']

    for w in words_to_remove:

        t = t.replace(w, ' ')

    t = re.sub(r'\s+', ' ', t).strip()

    return t



def parse_sku(product_name, sku_name):

    sku_cleaned = clean_text(sku_name)

    prod_cleaned = clean_text(product_name)

    full_lower = (str(sku_name) + ' ' + str(product_name)).lower()

    

    # 1. Trích xuất Size

    size = None

    # Tìm trong skuName đã làm sạch

    size_match = re.search(r'\b(nb|s|m|l|xl|xxl|xxxl|xxxxl)\b', sku_cleaned)

    if size_match:

        size = size_match.group(1).upper()

    else:

        # Tìm các mẫu dính liền như l54, m46

        size_num_match = re.search(r'\b(nb|s|m|l|xl|xxl|xxxl|xxxxl)(\d+)\b', sku_cleaned)

        if size_num_match:

            size = size_num_match.group(1).upper()

        else:

            # Tìm trong productName đã làm sạch

            size_prod_match = re.search(r'\b(nb|s|m|l|xl|xxl|xxxl|xxxxl)\b', prod_cleaned)

            if size_prod_match:

                size = size_prod_match.group(1).upper()



    # 2. Trích xuất Type (PANTS/TAPE)

    diaper_type = None

    sku_lower = str(sku_name).lower()

    prod_lower = str(product_name).lower()

    if any(k in sku_lower for k in ['dán', 'tape']):

        diaper_type = 'TAPE'

    elif any(k in sku_lower for k in ['quần', 'pants', 'pant']):

        diaper_type = 'PANTS'

    if diaper_type is None:

        if any(k in prod_lower for k in ['quần', 'pants', 'pant']) and not any(k in prod_lower for k in ['dán', 'tape']):

            diaper_type = 'PANTS'

        elif any(k in prod_lower for k in ['dán', 'tape']) and not any(k in prod_lower for k in ['quần', 'pants', 'pant']):

            diaper_type = 'TAPE'

        else:

            diaper_type = 'PANTS'

        

    # 3. Trích xuất Pack (Số miếng)

    pack = None

    # Kiểm tra mẫu phép cộng ví dụ 38+6

    add_match = re.search(r'(\d+)\s*\+\s*(\d+)', sku_cleaned)

    if add_match:

        pack = int(add_match.group(1)) + int(add_match.group(2))

    else:

        # Kiểm tra mẫu dính liền dạng m64

        size_num_match = re.search(r'\b(nb|s|m|l|xl|xxl|xxxl|xxxxl)(\d+)\b', sku_cleaned)

        if size_num_match:

            pack = int(size_num_match.group(2))

        else:

            # Tìm số đứng độc lập trong skuName

            standalone_match = re.findall(r'\b(\d+)\b', sku_cleaned)

            if standalone_match:

                valid_packs = [int(v) for v in standalone_match if 10 <= int(v) <= 150]

                if valid_packs:

                    pack = valid_packs[0]

                else:

                    # Gói dùng thử 1 cái

                    if '1' in standalone_match:

                        pack = 1

            

    # Nếu skuName không có pack, tìm trong productName

    if pack is None:

        add_match_prod = re.search(r'(\d+)\s*\+\s*(\d+)', prod_cleaned)

        if add_match_prod:

            pack = int(add_match_prod.group(1)) + int(add_match_prod.group(2))

        elif size:

            size_pattern = r'\b' + re.escape(size.lower()) + r'(\d+)\b'

            prod_size_match = re.search(size_pattern, prod_cleaned)

            if prod_size_match:

                pack = int(prod_size_match.group(1))

            else:

                # Bảng trung vị số miếng mặc định nếu hoàn toàn không tìm thấy

                default_packs = {

                    'NB': 90, 'S': 82, 'M': 64, 'L': 54, 'XL': 44, 'XXL': 36, 'XXXL': 46, 'XXXXL': 40

                }

                pack = default_packs.get(size, 50)

                

    # 4. Trích xuất khoảng cân nặng (MinKG, MaxKG)

    min_kg, max_kg = None, None

    weight_match = re.search(r'(\d+)\s*-\s*(\d+)\s*kg', full_lower)

    if weight_match:

        min_kg = float(weight_match.group(1))

        max_kg = float(weight_match.group(2))

    else:

        # Nếu khuyết, sử dụng bảng mặc định theo size

        if size in size_weight_lookup:

            min_kg, max_kg = size_weight_lookup[size]

            

    return size, diaper_type, pack, min_kg, max_kg


In [4]:
print("Đang chạy parser trích xuất thuộc tính trên danh sách biến thể...")

parsed_rows = []

for idx, row in df_skus.iterrows():

    size, dtype, pack, min_kg, max_kg = parse_sku(row['productName'], row['skuName'])

    parsed_rows.append({

        'itemId': row['itemId'],

        'modelId': row['modelId'],

        'productName': row['productName'],

        'skuName': row['skuName'],

        'size': size,

        'type': dtype,

        'pack': pack,

        'min_kg': min_kg,

        'max_kg': max_kg

    })



df_parsed = pd.DataFrame(parsed_rows)

print(f"Hoàn thành trích xuất đặc trưng cho {len(df_parsed)} SKU.")


Đang chạy parser trích xuất thuộc tính trên danh sách biến thể...
Hoàn thành trích xuất đặc trưng cho 157 SKU.


In [5]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===\n")



total_skus = len(df_parsed)

null_sizes = df_parsed['size'].isnull().sum()

null_types = df_parsed['type'].isnull().sum()

null_packs = df_parsed['pack'].isnull().sum()

null_min_kg = df_parsed['min_kg'].isnull().sum()



success_rate = (1 - (null_sizes / total_skus)) * 100

print(f"1. Tỷ lệ trích xuất thành công Size: {success_rate:.2f}% (Yêu cầu >= 95%)")

assert success_rate >= 95.0, f"Lỗi: Tỷ lệ thành công chỉ đạt {success_rate:.2f}%"



# Đảm bảo không có giá trị khuyết thiếu trong các đặc trưng mới

assert null_types == 0, "Lỗi: Phát hiện giá trị rỗng ở type"

assert null_packs == 0, "Lỗi: Phát hiện giá trị rỗng ở pack"

assert null_min_kg == 0, "Lỗi: Phát hiện giá trị rỗng ở min_kg"

print("2. Đảm bảo các cột đặc trưng mới sinh ra không chứa giá trị Null: Đạt")



# Đảm bảo các giá trị nằm trong phạm vi cho phép

valid_sizes = ['NB', 'S', 'M', 'L', 'XL', 'XXL', 'XXXL', 'XXXXL']

valid_types = ['PANTS', 'TAPE']

assert df_parsed['size'].isin(valid_sizes).all(), "Lỗi: Phát hiện size ngoài danh mục cho phép"

assert df_parsed['type'].isin(valid_types).all(), "Lỗi: Phát hiện type ngoài danh mục cho phép"

print("3. Không có giá trị không hợp lệ ngoài danh mục cho phép: Đạt")



print("\n4. In ngẫu nhiên 30 SKU tã/bỉm đã trích xuất thành công để kiểm định thủ công:")

print(df_parsed[['skuName', 'size', 'type', 'pack', 'min_kg', 'max_kg']].sample(30, random_state=42).to_string())


=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===

1. Tỷ lệ trích xuất thành công Size: 100.00% (Yêu cầu >= 95%)
2. Đảm bảo các cột đặc trưng mới sinh ra không chứa giá trị Null: Đạt
3. Không có giá trị không hợp lệ ngoài danh mục cho phép: Đạt

4. In ngẫu nhiên 30 SKU tã/bỉm đã trích xuất thành công để kiểm định thủ công:
                                    skuName size   type  pack  min_kg  max_kg
126              Supdry Xanh,L quần - 1 cái    L  PANTS     1     9.0    14.0
45   Cộng Miếng (Xanh),Nữa bịch NB dùng thử   NB  PANTS    90     0.0     5.0
133              Supdry Trắng,S dán - 1 cái    S   TAPE     1     4.0     8.0
138                            Tã Dán - S82    S   TAPE    82     4.0     8.0
111                                   XXL26  XXL  PANTS    26    15.0    25.0
29                            Size M74 quần    M  PANTS    74     6.0    11.0
95                 S58 tã dán (4-8kg),Moony    S   TAPE    58     4.0     8.0
97                 Nb63 tã dán (~5kg),Moony   N

In [6]:
print("=== LƯU TRỮ DỮ LIỆU THUỘC TÍNH TÃ BỈM ===")

attributes_path = os.path.join(DATA_DIR, 'diaper_skus_attributes.csv')

df_parsed.to_csv(attributes_path, index=False)

print(f"Đã lưu kết quả thành công tại: {attributes_path}")


=== LƯU TRỮ DỮ LIỆU THUỘC TÍNH TÃ BỈM ===
Đã lưu kết quả thành công tại: /home/nguyen-dai-tung/supplyforge-ai/test-data/Database - NCKH/diaper_skus_attributes.csv
